In [ ]:
# Loading Libraries
from itertools import combinations
import pandas as pd
import numpy as np
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType
from databricks.feature_engineering import FeatureEngineeringClient


In [ ]:
# --- Permutation entropy (payment & debt) ---
def weighted_permutation_entropy(matrix: np.ndarray, order=3) -> np.ndarray:
    """
    Calculates Weighted Permutation Entropy for a time-series matrix using sliding windows.
    Measures the complexity/unpredictability of a user's financial habits.
    
    With only 6 time points, this serves as a localized heuristic for 
    behavioral volatility rather than a true asymptotic entropy measure, 
    but it provides strong predictive signal for tree-based models.
    """
    #adding noise to break ties
    row_std = np.std(matrix, axis=1)
    is_constant = row_std < 1e-8
    rng = np.random.RandomState(67)
    noise_scale = row_std * 1e-8
    noise_scale[is_constant] = 0
    noise = rng.uniform(0, 1, size=matrix.shape) * noise_scale[:, np.newaxis]
    matrix = matrix + noise
    windows = np.lib.stride_tricks.sliding_window_view(matrix, window_shape=order, axis=1)
    ranks = np.argsort(np.argsort(windows, axis=2), axis=2)
    powers = order ** np.arange(order - 1, -1, -1)
    hashes = np.sum(ranks * powers, axis=2)

    # Amplitude weights: variance within each window
    window_means = np.mean(windows, axis=2, keepdims=True)
    amp_weights = np.sum((windows - window_means) ** 2, axis=2)  # (n, W)

     # Aggregate weighted probabilities per pattern
    unique_patterns = np.unique(hashes)
    mask_3d = (hashes[:, :, None] == unique_patterns[None, None, :])  # (n, W, P)
    weighted_probs = np.sum(amp_weights[:, :, None] * mask_3d, axis=1)  # (n, P)
    total = np.sum(weighted_probs, axis=1, keepdims=True)
    total = np.maximum(total, 1e-10)
    weighted_probs /= total
    # ---------------------------------------------------------
    # Max entropy in theory should be 3!
    # but since we only have 6 time points, the log of the maximum number of windows 
    # which is 4 is the maximum entropy
    # ---------------------------------------------------------
    log_p = np.where(weighted_probs > 0, np.log2(weighted_probs), 0)# convention: 0·log(0) = 0
    wpe = -np.sum(weighted_probs * log_p, axis=1)
    max_entropy = np.log2(windows.shape[1])
    normalized_entropy = wpe / max_entropy
    normalized_entropy[is_constant] = 0
    return normalized_entropy
    pass


In [ ]:
# Building the features
def generate_credit_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    This function takes a chunk of raw data (pdf), calculates all the math, 
    and returns a clean table of just the User IDs and their new features.
    """

    # =============================================================================
    # 1. CORE MATRICES
    # =============================================================================

    pay_statuses = ['PAY_1', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
    debts = ['BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6']
    payments = ['PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']

    baseline = (0.005 * df['LIMIT_BAL']).values

    debt_matrix = df[debts].values
    payment_matrix = df[payments].values
    pay_matrix = df[pay_statuses].values
    bills = debt_matrix[:, 1:]
    safe_bills = np.maximum(bills, baseline[:, np.newaxis])
    credit_balance_mask = bills < 0
    lagged_payments = payment_matrix[:, :-1]

    # Chronological orderings (oldest → newest)
    chronological_debt = debt_matrix[:, ::-1]
    chronological_payments = payment_matrix[:, ::-1]
    chronological_pay_statuses = pay_matrix[:, ::-1]

    # Common decay weights (half-life = 2 months)
    half_life = 2
    decay = np.log(2) / half_life

    features_df = pd.DataFrame({'ID': df['ID']})

    # =============================================================================
    # 2. PAYOFF RATIO FEATURES
    # =============================================================================

    payoff_ratio_matrix = np.clip(lagged_payments / safe_bills, None, 2)
    payoff_ratio_matrix[credit_balance_mask] = 1
    payoff_ratio_matrix[np.isnan(payoff_ratio_matrix)] = 0

    payoff_ratios = [f"payoff_ratio_{i}" for i in range(1, 6)]
    features_df[payoff_ratios] = pd.DataFrame(payoff_ratio_matrix, index=features_df.index)
    features_df['mean_payoff_ratio'] = np.mean(payoff_ratio_matrix, axis=1)
    features_df['full_payment_count'] = np.sum(payoff_ratio_matrix >= 0.95, axis=1)
    features_df['micro_payment_count'] = np.sum((payoff_ratio_matrix > 0) & (payoff_ratio_matrix <= 0.1), axis=1)
    features_df['zero_payment_count'] = np.sum(payment_matrix == 0, axis=1)

    # =============================================================================
    # 3. EXPONENTIALLY WEIGHTED MOVING AVERAGES (EWMA)
    # =============================================================================

    weights_6 = np.exp(-decay * np.arange(6))
    weights_6 /= weights_6.sum()

    features_df["ewma_payments"] = payment_matrix @ weights_6
    features_df["ewma_debt"] = debt_matrix @ weights_6

    ewma_debt = features_df['ewma_debt'].values
    ewma_payments = features_df['ewma_payments'].values
    no_debt_mask = ewma_debt <= 0
    safe_ewma_debt = np.maximum(ewma_debt, baseline)
    ratio = np.clip(ewma_payments / safe_ewma_debt, None, 2)
    ratio[no_debt_mask] = 1
    features_df["ewma_payment_to_bill_ratios"] = ratio

    features_df['ewma_utilization'] = features_df['ewma_debt'] / df['LIMIT_BAL']

    # =============================================================================
    # 4. DEBT MOMENTUM FEATURES
    # =============================================================================

    # --- Normalised debt momentum (Theil-Sen robust slopes) ---
    time_points_6 = np.arange(6)
    all_slopes = []
    for i, j in combinations(range(len(time_points_6)), 2):
        dt = time_points_6[j] - time_points_6[i]
        slope = (debt_matrix[:, i] - debt_matrix[:, j]) / dt
        all_slopes.append(slope)

    robust_slopes = np.median(np.array(all_slopes), axis=0)
    features_df["debt_momentem"] = robust_slopes / baseline

    # --- Exponentially weighted least squares (EWLS) debt momentum ---
    weights_6_momentum = np.exp(-decay * time_points_6)
    weights_6_momentum /= weights_6_momentum.sum()

    time_weighted_mean = np.sum(weights_6_momentum * time_points_6)
    time_weighted_diff = time_points_6 - time_weighted_mean
    debt_weighted_mean = np.sum(debt_matrix * weights_6_momentum, axis=1, keepdims=True)
    debt_weighted_diff = debt_matrix - debt_weighted_mean

    ewls_debt_momentum = -1 * (
        np.sum(weights_6_momentum * debt_weighted_diff * time_weighted_diff, axis=1)
        / np.sum(weights_6_momentum * time_weighted_diff ** 2)
    )
    features_df["ewls_debt_momentum"] = ewls_debt_momentum

    # =============================================================================
    # 5. DEBT ACCELERATION FEATURES
    # =============================================================================

    first_deriv = np.diff(chronological_debt, axis=1)
    second_deriv = np.diff(first_deriv, axis=1)

    debt_acceleration_mean = np.mean(second_deriv, axis=1)
    debt_acceleration_recent = second_deriv[:, -1]
    features_df['debt_accel_positive_count'] = np.sum(second_deriv > 0, axis=1)
    features_df['norm_debt_acceleration_mean'] = debt_acceleration_mean / df['LIMIT_BAL']

    # =============================================================================
    # 6. DEBT SPIKE & ANOMALY DETECTION
    # =============================================================================

    recent_debt = debt_matrix[:, 0]
    max_recent = np.max(debt_matrix[:, :2], axis=1)
    min_historical = np.min(debt_matrix[:, 2:], axis=1)
    safe_min_hist = np.maximum(min_historical, baseline)

    features_df['debt_spike_ratio'] = (max_recent - min_historical) / safe_min_hist

    # Recent vs median debt (MAD-based anomaly score)
    median_debt = np.median(debt_matrix, axis=1)
    mad_debt = np.median(np.abs(debt_matrix - np.median(debt_matrix, axis=1, keepdims=True)), axis=1)
    features_df['debt_anomaly_score'] = (recent_debt - median_debt) / np.maximum(mad_debt, baseline)

    # =============================================================================
    # 7. PAYMENT VOLATILITY FEATURES
    # =============================================================================

    features_df['payment_volatility'] = payment_matrix.std(axis=1)
    features_df['relative_payment_volatility'] = features_df['payment_volatility'] / np.maximum(payment_matrix.mean(axis=1), baseline)

    first_deriv_payments = np.diff(chronological_payments, axis=1)
    features_df['recent_payment_momentum'] = first_deriv_payments[:, -1]

    # =============================================================================
    # 8. CREDIT UTILIZATION FEATURES
    # =============================================================================

    features_df['ewls_utilization_burn_rate'] = features_df['ewls_debt_momentum'] / df['LIMIT_BAL']
    features_df['utilization_shock_recent'] = debt_acceleration_recent / df['LIMIT_BAL']
    features_df['max_credit_utilization'] = np.max(debt_matrix, axis=1) / df['LIMIT_BAL']
    features_df['recent_remaining_liquidity'] = np.maximum(df['LIMIT_BAL'] - debt_matrix[:, 0], 0)

    # =============================================================================
    # 9. DELINQUENCY FEATURES
    # =============================================================================

    pay_matrix = features_df[pay_statuses].values
    late_pay_matrix = np.maximum(pay_matrix, 0)

    features_df['ewma_pay_status'] = late_pay_matrix @ weights_6
    features_df['max_delinquency'] = np.max(late_pay_matrix, axis=1)
    features_df['delinquency_severity'] = np.sum(late_pay_matrix, axis=1)
    features_df['delinquency_frequency'] = np.sum(late_pay_matrix > 0, axis=1)
    features_df['delinquency_shift'] = np.maximum(df['PAY_1'], 0) - np.maximum(df['PAY_6'], 0)

    is_delinquent = pay_matrix > 0
    first_delinquency = np.argmax(is_delinquent, axis=1)
    features_df['months_since_last_delinquency'] = np.where(is_delinquent.any(axis=1), first_delinquency, 6)

    late_pay_reversed = late_pay_matrix[:, ::-1]
    escalations = np.diff(late_pay_reversed, axis=1)
    positive_escalations = np.maximum(escalations, 0)
    features_df['max_delinquency_escalation'] = np.max(positive_escalations, axis=1)

    # =============================================================================
    # 10. AGE-RELATED FEATURES
    # =============================================================================

    features_df['limit_to_age_ratio'] = df['LIMIT_BAL'] / df['AGE']
    features_df['age_adjusted_utilization'] = features_df['ewma_utilization'] / df['AGE']

    # =============================================================================
    # 11. NEW SPEND FEATURES
    # =============================================================================

    bills_for_spend = debt_matrix[:, :-1]
    previous_bills = debt_matrix[:, 1:]
    new_spends_matrix = bills_for_spend - (previous_bills - lagged_payments)

    new_spends = [f'new_spends{i}' for i in range(1, 6)]
    features_df[new_spends] = pd.DataFrame(new_spends_matrix, index=features_df.index)
    features_df['total_new_spend'] = new_spends_matrix.sum(axis=1)
    features_df['total_spend_to_limit_ratio'] = features_df['total_new_spend'] / df['LIMIT_BAL']

    # EWMA of spends
    time_points_5 = np.arange(5)
    spend_weights = np.exp(-decay * time_points_5)
    spend_weights /= spend_weights.sum()
    features_df['ewma_spends'] = new_spends_matrix @ spend_weights

    # EWLS spend momentum
    weights_5 = np.exp(-decay * time_points_5)
    weights_5 /= weights_5.sum()
    time_weighted_mean_5 = np.sum(weights_5 * time_points_5)
    time_weighted_diff_5 = time_points_5 - time_weighted_mean_5
    spend_weighted_mean = np.sum(new_spends_matrix * weights_5, axis=1, keepdims=True)
    spend_weighted_diff = new_spends_matrix - spend_weighted_mean
    ewls_spend_momentum = -1 * (
        np.sum(weights_5 * spend_weighted_diff * time_weighted_diff_5, axis=1)
        / np.sum(weights_5 * time_weighted_diff_5 ** 2)
    )
    features_df['ewls_spend_momentum'] = ewls_spend_momentum

    median_spend = np.median(new_spends_matrix, axis=1)
    features_df['norm_ewls_spend_momentum'] = features_df['ewls_spend_momentum'] / df['LIMIT_BAL']
    features_df['ewma_spend_to_pay_ratio'] = features_df['ewma_spends'] / np.maximum(features_df['ewma_payments'], baseline)

    mad_spend = np.median(np.abs(new_spends_matrix - np.median(new_spends_matrix, axis=1, keepdims=True)), axis=1)

    # =============================================================================
    # 14. ACTIVITY & VARIATION FEATURES
    # =============================================================================

    features_df['active_debt_months'] = np.sum(debt_matrix != 0, axis=1)
    features_df['active_spend_months'] = np.sum(new_spends_matrix != 0, axis=1)

    consecutive_zeros = np.argmax(payment_matrix != 0, axis=1)
    features_df['consecutive_zero_payments'] = np.where((payment_matrix != 0).any(axis=1), consecutive_zeros, 6)

    features_df['debt_no_variation'] = (mad_debt == 0).astype(int)
    features_df['spend_no_variation'] = (mad_spend == 0).astype(int)

    features_df['spend_anomaly_score'] = (new_spends_matrix[:, 0] - median_spend) / np.maximum(mad_spend, baseline)
    features_df['credit_balance_count'] = np.sum(debt_matrix < 0, axis=1)

    # =============================================================================
    # 15. ENTROPY FEATURES
    # =============================================================================

    # --- Pay status transition entropy (Shannon entropy of state changes) ---
    pay_transitions = np.diff(chronological_pay_statuses, axis=1)
    unique_vals = np.unique(pay_matrix)
    trans_counts = np.sum(pay_transitions[:, :, None] == unique_vals[None, None, :], axis=1) # broadcasting
    probs = trans_counts / pay_transitions.shape[1]
    log_probs = np.where(probs > 0, np.log2(probs), 0)
    features_df['pay_status_entropy'] = -np.sum(probs * log_probs, axis=1)

    features_df['payment_entropy'] = weighted_permutation_entropy(payment_matrix)
    features_df['debt_entropy'] = weighted_permutation_entropy(debt_matrix)

    # =============================================================================
    # 15. PAY STATUS REVERSAL
    # =============================================================================

    diffs = np.diff(chronological_pay_statuses, axis=1)
    signs = np.sign(diffs)

    # ---------------------------------------------------------
    # MOMENTUM FORWARD-FILL TO HANDLE FLATLINES:
    # If a user's status doesn't change for a month (sign == 0), we don't want to 
    # treat that as a behavioral reversal. Instead, we forward-fill the previous 
    # month's momentum. This ensures that pausing and continuing in the same 
    # direction isn't falsely flagged as a flip-flop.
    # ---------------------------------------------------------
    for col in range(1, signs.shape[1]):
        mask = signs[:, col] == 0
        signs[mask, col] = signs[mask, col - 1]

    changes = (signs[:, 1:] != signs[:, :-1])
    valid = (signs[:, :-1] != 0) & (signs[:, 1:] != 0)
    features_df['pay_status_reversals'] = (changes & valid).sum(axis=1)

    # =============================================================================
    # 16. BEHAVIORAL SHIFT FEATURES
    # =============================================================================

    recent_payoff = np.mean(payoff_ratio_matrix[:, :2], axis=1)
    historical_payoff = np.mean(payoff_ratio_matrix[:, 2:], axis=1)
    features_df['payoff_behavior_shift'] = recent_payoff - historical_payoff

    recent_spend = np.mean(new_spends_matrix[:, :2], axis=1)
    historical_spend = np.mean(new_spends_matrix[:, 2:], axis=1)
    features_df['spend_behavior_shift'] = (recent_spend - historical_spend) / np.maximum(df['LIMIT_BAL'], baseline)
    return features_df


In [ ]:
# Feature wrapper
def spark_feature_wrapper(iterator):
    """
    Takes a stream of raw Pandas DataFrames from Spark, 
    processes each one using your recipe, and streams the results back.
    """
    for df in iterator:
        features_df = generate_credit_features(df)
        yield features_df


In [ ]:
# Feature table schema
feature_schema = StructType([
    # --- Primary Key ---
    StructField("ID", IntegerType(), True),

    # --- 1. Payoff Ratio Features ---
    StructField("payoff_ratio_1", DoubleType(), True),
    StructField("payoff_ratio_2", DoubleType(), True),
    StructField("payoff_ratio_3", DoubleType(), True),
    StructField("payoff_ratio_4", DoubleType(), True),
    StructField("payoff_ratio_5", DoubleType(), True),
    StructField("mean_payoff_ratio", DoubleType(), True),
    StructField("full_payment_count", IntegerType(), True),
    StructField("micro_payment_count", IntegerType(), True),
    StructField("zero_payment_count", IntegerType(), True),

    # --- 2. EWMA Features ---
    StructField("ewma_payments", DoubleType(), True),
    StructField("ewma_debt", DoubleType(), True),
    StructField("ewma_payment_to_bill_ratios", DoubleType(), True),
    StructField("ewma_utilization", DoubleType(), True),

    # --- 3 & 4. Debt Momentum & Acceleration ---
    StructField("debt_momentem", DoubleType(), True), # Note: kept your spelling "momentem"
    StructField("ewls_debt_momentum", DoubleType(), True),
    StructField("debt_accel_positive_count", IntegerType(), True),
    StructField("norm_debt_acceleration_mean", DoubleType(), True),

    # --- 5. Debt Spikes & Anomalies ---
    StructField("debt_spike_ratio", DoubleType(), True),
    StructField("debt_anomaly_score", DoubleType(), True),

    # --- 6. Payment Volatility ---
    StructField("payment_volatility", DoubleType(), True),
    StructField("relative_payment_volatility", DoubleType(), True),
    StructField("recent_payment_momentum", DoubleType(), True),

    # --- 7. Utilization ---
    StructField("ewls_utilization_burn_rate", DoubleType(), True),
    StructField("utilization_shock_recent", DoubleType(), True),
    StructField("max_credit_utilization", DoubleType(), True),
    StructField("recent_remaining_liquidity", DoubleType(), True),

    # --- 8. Delinquency ---
    StructField("ewma_pay_status", DoubleType(), True),
    StructField("max_delinquency", IntegerType(), True),
    StructField("delinquency_severity", IntegerType(), True),
    StructField("delinquency_frequency", IntegerType(), True),
    StructField("delinquency_shift", IntegerType(), True),
    StructField("months_since_last_delinquency", IntegerType(), True),
    StructField("max_delinquency_escalation", IntegerType(), True),

    # --- 9. Age-Related ---
    StructField("limit_to_age_ratio", DoubleType(), True),
    StructField("age_adjusted_utilization", DoubleType(), True),

    # --- 10. New Spend ---
    StructField("new_spends1", DoubleType(), True),
    StructField("new_spends2", DoubleType(), True),
    StructField("new_spends3", DoubleType(), True),
    StructField("new_spends4", DoubleType(), True),
    StructField("new_spends5", DoubleType(), True),
    StructField("total_new_spend", DoubleType(), True),
    StructField("total_spend_to_limit_ratio", DoubleType(), True),
    StructField("ewma_spends", DoubleType(), True),
    StructField("ewls_spend_momentum", DoubleType(), True),
    StructField("norm_ewls_spend_momentum", DoubleType(), True),
    StructField("ewma_spend_to_pay_ratio", DoubleType(), True),

    # --- 11. Activity Indicators ---
    StructField("active_debt_months", IntegerType(), True),
    StructField("active_spend_months", IntegerType(), True),
    StructField("consecutive_zero_payments", IntegerType(), True),
    StructField("debt_no_variation", IntegerType(), True),
    StructField("spend_no_variation", IntegerType(), True),
    StructField("spend_anomaly_score", DoubleType(), True),
    StructField("credit_balance_count", IntegerType(), True),

    # --- 12 Entropy & Shifts ---
    StructField("pay_status_entropy", DoubleType(), True),
    StructField("payment_entropy", DoubleType(), True),
    StructField("debt_entropy", DoubleType(), True),
    StructField("pay_status_reversals", IntegerType(), True),
    StructField("payoff_behavior_shift", DoubleType(), True),
    StructField("spend_behavior_shift", DoubleType(), True)
])        


In [ ]:
# Saving the features
raw_spark_df = spark.table("workspace.default.credit_model_data")
distributed_features_df = raw_spark_df.mapInPandas(spark_feature_wrapper, schema=feature_schema)
fe = FeatureEngineeringClient()
fe.create_table(
    name="workspace.ml_features.engineered_features",
    primary_keys=["ID"],
    df=distributed_features_df,
    description="Engineered Features"
)